<a href="https://colab.research.google.com/github/davisdefoe/machine-learning/blob/main/program%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files

# 1. Upload your Kaggle dataset CSV file
print("Please upload your Kaggle dataset CSV file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 2. Load and Preprocess Data
df = pd.read_csv(filename)
print(f"\nDataset loaded successfully! Shape: {df.shape}")

# Drop an 'Id' column if it exists in the dataset
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

# Separate features and target
X = df.iloc[:, :-1].values
y_raw = df.iloc[:, -1].values

# Automatically convert text labels (strings) to numbers
if y_raw.dtype == object or isinstance(y_raw[0], str):
    y_raw, class_names = pd.factorize(y_raw)
    print(f"Detected categorical classes: {class_names}")

# Reshape target to a column vector
y = y_raw.reshape(-1, 1).astype(float)

# Normalize features
scaler_x = StandardScaler()
X = scaler_x.fit_transform(X)

# Scale target between 0 and 1 for the sigmoid output layer
if np.max(y) > 0:
    y = y / np.max(y)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Activation Functions and Derivatives
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(output):
    return output * (1 - output)

# 4. Neural Network Class with Backpropagation
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.lr = learning_rate
        # Xavier weight initialization
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = sigmoid(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def backward(self, X, y, output):
        m = X.shape[0]

        # Calculate output layer error and gradients
        error_output = output - y
        d_output = error_output * sigmoid_derivative(output)

        # Calculate hidden layer error and gradients
        error_hidden = np.dot(d_output, self.W2.T)
        d_hidden = error_hidden * sigmoid_derivative(self.A1)

        # Compute weight and bias gradients
        dW2 = np.dot(self.A1.T, d_output) / m
        db2 = np.sum(d_output, axis=0, keepdims=True) / m
        dW1 = np.dot(X.T, d_hidden) / m
        db1 = np.sum(d_hidden, axis=0, keepdims=True) / m

        # Update weights and biases
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=1500):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output)

            if (epoch + 1) % 200 == 0:
                loss = np.mean(np.square(output - y))
                print(f"Epoch {epoch+1}/{epochs} - Mean Squared Error Loss: {loss:.6f}")

# 5. Initialize and Train the Model
input_neurons = X_train.shape[1]
hidden_neurons = 6
output_neurons = y_train.shape[1]

nn = NeuralNetwork(input_size=input_neurons, hidden_size=hidden_neurons, output_size=output_neurons, learning_rate=0.5)

print("\nTraining Neural Network...")
nn.train(X_train, y_train, epochs=1500)

# 6. Evaluate on Test Data
predictions = nn.forward(X_test)
test_loss = np.mean(np.square(predictions - y_test))
print(f"\nEvaluation Complete! Test Mean Squared Error: {test_loss:.6f}")

Please upload your Kaggle dataset CSV file:


Saving heart disease.csv to heart disease.csv

Dataset loaded successfully! Shape: (1025, 14)

Training Neural Network...
Epoch 200/1500 - Mean Squared Error Loss: 0.141831
Epoch 400/1500 - Mean Squared Error Loss: 0.112336
Epoch 600/1500 - Mean Squared Error Loss: 0.103175
Epoch 800/1500 - Mean Squared Error Loss: 0.099148
Epoch 1000/1500 - Mean Squared Error Loss: 0.096920
Epoch 1200/1500 - Mean Squared Error Loss: 0.095341
Epoch 1400/1500 - Mean Squared Error Loss: 0.093942

Evaluation Complete! Test Mean Squared Error: 0.143851
